# Código de Validación de Resultados

Este script toma un **Excel** con inscripciones del CBR (una fila por inscripción) que contiene el **texto fuente** en la columna `contenido` y varios **campos ya extraídos** (p. ej. `rut_emisor`, `fojas_cbr_anterior`). Para cada fila, hace una **validación asistida por IA** en tres pasos:

<br>


1. **Recuperación de contexto con embeddings (RAG liviano):**

   * Divide el texto `contenido` en *chunks* (trozos) solapados.
   * Calcula embeddings (`text-embedding-3-small`) para cada chunk.
   * Para cada campo a validar, usa una “consulta” (`FIELD_QUERIES`) y recupera los **top_k** chunks más relevantes.
   * Con eso construye un “contexto por campo” (`[CONTEXTO_<campo>] ... [/CONTEXTO_<campo>]`) para reducir tokens y focalizar la validación.

<br>

2. **Validación con modelo GPT usando salida estructurada (Pydantic parse):**

   * Envía el contexto recuperado + los valores originales (payload) al modelo `gpt-5-mini`.
   * Exige una respuesta **estructurada y validable** mediante un modelo Pydantic dinámico (`build_pydantic_response_model`).
   * Para cada campo devuelve:

     * `<campo>_status`: `OK | PARCIAL | WRONG | MISSING`
     * `<campo>_valor_texto`: el valor “correcto” re-extraído y normalizado
     * `<campo>_comentario`: explicación breve

<br>

3. **Salida y métricas:**

   * Agrega estas columnas al DataFrame y guarda un Excel de salida con:

     * `detalle_validacion`: datos originales + resultados de validación
     * `resumen_precision`: precisión por campo (porcentaje de `OK` sobre filas evaluables)

Además, carga la API key desde un archivo `.env` (`DOTENV_PATH`) usando la variable `cnep-openai-2` (o `OPENAI_API_KEY` como alternativa), y contiene manejo de errores: si falla la llamada a IA, marca el campo como `MISSING` con el motivo en el comentario.


## Librerías

In [41]:
# --- Librerías estándar ---
import os
import json
import re
import time
import copy
import shutil
from itertools import permutations
from html import unescape
from glob import glob
import pandas as pd
import numpy as np
from tqdm import tqdm

# --- IA ---
from dotenv import load_dotenv
import openai
from openai import OpenAI
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.schema import Document
from langchain_openai import ChatOpenAI
from langchain.prompts import ChatPromptTemplate

# --- Pydantic ---
from enum import Enum 
from typing import List, Optional, Literal, Dict, Union, Annotated, Any, Tuple
from pydantic import BaseModel, Field, ValidationError, field_validator, ConfigDict, constr, conint, conlist, model_validator, create_model
from pydantic.types import StringConstraints

## Paths y API Keys

In [44]:
path = os.path.join(os.environ['USERPROFILE'], "RUTA")

# Cargar API KEY
load_dotenv(dotenv_path=os.path.join(path, "Códigos",  ".env"))
api_key_openai = os.getenv("OPENAI_API_KEY")
client = openai.OpenAI(api_key=api_key_openai)

## Carga de Datos

In [47]:
df = pd.read_excel(os.path.join(path, "Resultados", "Extracción", "resultados.xlsx"))

### Listado de Conservadores

In [52]:
df_cbr = pd.read_excel(os.path.join(path, "Archivos Auxiliares", "Listados Auxiliares.xlsx"), sheet_name="Listado CBRs")
Region = Enum("Region", {row["region"]: row["nombre_region"] for _, row in df_cbr.drop_duplicates("region").iterrows()}, type=str)
Conservador = Enum("Conservador", {row["conservador"]: row["nombre_conservador"] for _, row in df_cbr.drop_duplicates("conservador").iterrows()}, type=str)
region_por_conservador = {row["nombre_conservador"]: row["nombre_region"] for _, row in df_cbr.iterrows()}

### Listado Variables

In [55]:
variables = pd.read_excel(os.path.join(path, "Archivos Auxiliares", "Listados Auxiliares.xlsx"), sheet_name="Descripción de Variables")

### Listado Tipos de Transacción

In [58]:
df_config = pd.read_excel(os.path.join(path, "Archivos Auxiliares", "Listados Auxiliares.xlsx"), sheet_name="Tipos de Transacción")

## Modelo de datos

In [61]:
path_modelo = (os.path.join(path, "Códigos", "Extracción", "Modelo_Datos.ipynb"))
%run "{path_modelo}"

## Instrucciones

In [64]:
path_instrucciones = (os.path.join(path, "Códigos", "Extracción", "Instrucciones.ipynb"))
%run "{path_instrucciones}"

## Funciones

In [67]:
def gpt_call(client, sys_prompt, instr_fase, resp_fase, texto_auditar, respuesta_auditar,  response_model):
    """Función de ejecución simple"""
    try:
        completion = client.beta.chat.completions.parse(
            model="gpt-5-mini",
            messages=[
                {"role": "system", "content": sys_prompt},
                {"role": "system", "content": f"INSTRUCCIONES UTILIZADAS EN LA FASE DE EXTRACCIÓN:\n\"\"\"{instr_fase}\"\"\""},
                {"role": "system", "content": f"FORMATO DE RESPUESTA UTILIZADO EN LA FASE DE EXTRACCIÓN:\n\"\"\"{resp_fase}\"\"\""},
                {"role": "user", "content": f"DOCUMENTO A AUDITAR:\n\"\"\"{texto_auditar}\"\"\""},
                {"role": "user", "content": f"RESPUESTA A AUDITAR:\n\"\"\"{respuesta_auditar}\"\"\""}
            ],
            response_format=response_model
        )
        return completion.choices[0].message.parsed
    except Exception as e:
        print(f"Error API: {e}")
        return None

def normalizar_str(x: Any) -> str:
    if pd.isna(x): return ""
    return str(x).strip()

In [69]:
Status = Literal["OK", "PARCIAL", "WRONG"]

def build_pydantic_model_dynamic(columns: List[str]):
    """Crea la clase Pydantic dinámicamente para un subconjunto de columnas"""
    fields = {}
    for c in columns:
        fields[f"{c}_status"] = (Status, Field(..., description=f"Estado validación para {c}"))
        fields[f"{c}_valor_texto"] = (str, Field(..., description=f"Valor hallado en texto para {c}"))
        fields[f"{c}_comentario"] = (str, Field(..., description=f"Justificación para {c}"))
    
    Resp = create_model("FormatoRespuesta", __config__=ConfigDict(extra="forbid"), **fields)
    
    return Resp

In [71]:
sys_prompt = """
ROL:
Eres un Auditor Experto en Derechos de Agua y Registros del Conservador de Bienes Raíces de Chile.

OBJETIVO:
Validar si las variables extraídas desde un texto legal son correctas, de acuerdo con:
- Las instrucciones utilizadas en la fase de extracción.
- El formato de respuesta definido para dicha fase.
- El contenido explícito del documento legal.

INSUMOS QUE RECIBIRÁS:
1. INSTRUCCIONES UTILIZADAS EN LA FASE DE EXTRACCIÓN
2. FORMATO DE RESPUESTA UTILIZADO EN LA FASE DE EXTRACCIÓN
3. DOCUMENTO A AUDITAR (texto legal)
4. RESPUESTA A AUDITAR (resultado de la extracción)

TAREA:
Para CADA variable definida en el formato de respuesta, debes:
- Evaluar si el contenido reportado en la RESPUESTA A AUDITAR es correcto según el contenido de DOCUMENTO A AUDITAR, y las instrucciones y formato. 
- Completar correctamente el "valor_texto", que es el contenido que debiese estar en la respuesta a auditar

CRITERIOS PARA "status"
Usa exclusivamente uno de los siguientes valores:

- OK: El valor reportado coincide semánticamente con el texto y las instrucciones, o tiene diferencias poco relevantes.
- PARCIAL: El valor es parcialmente correcto, incompleto o impreciso y sería fundamental corregir.
- WRONG: El texto indica explícitamente un valor distinto al reportado. Se está extrayendo la información de una manera muy incorrecta conforme a las instrucciones y el texto. 

Criterios fundamentales:
- 'NaN', 'null', '', 'None', '0','['']', '[]','NO INDICA/OTRO', etc. Son todos valores que debes considerar equivalentes. 
   Por ejemplo hay un 'NaN' y estimas que debería decir 'NO INDICA/OTRO' el status es OK
- Algunos de los campos a estudiar contienen LISTAS de elementos. Para dichas variables revisa si el CONTENIDO es el correcto. El ORDEN de los elementos listados es IRRELEVANTE.
  Si una lista tiene elementos en un orden incorrecto pero su contenido fue extraído de manera correcta, el status es OK. 

CRITERIOS PARA "comentario": justificar la elección del "status" y el "valor_texto" de dicha variable. 
- Si "status" es "OK" el comentario estará vacío, y el "valor_texto" será igual al reportado en la respuesta a auditar. 
"""


## Ejecución

In [74]:
config_fases = []

for i in range(1, 8):
    config_fases.append({
        "id": f"fase_{i}",
        "formato_respuesta": globals()[f"Fase{i}Registro"],
        "instrucciones": globals()[f"instr_fase{i}"],
        "columnas": variables.loc[variables['modelo'] == i, 'nombre_columna'].tolist()
    })

In [76]:
# --- LOOP POR DOCUMENTO ---
for idx, fila in df[:].iterrows():
    print(f"\nProcesando Documento ID: {idx}...")

    ## Definir Texto a Auditar
    texto_auditar = normalizar_str(fila.get("contenido", "")) 
    
    # --- LOOP MODELOS ---
    for fase in config_fases[:]:
        # Extraemos la información de la configuración de la fase actual
        model_id = fase["id"]
        instr_fase = fase["instrucciones"]
        formato_respuesta = fase["formato_respuesta"]
        columnas_modelo = fase["columnas"]

        ## Definir Resp Fase
        resp_fase = json.dumps(formato_respuesta.model_json_schema(), indent=2, ensure_ascii=False)
        
        ## Definir Respuesta a Auditar 
        respuesta_auditar = "\n".join([f"- {col}: {normalizar_str(str(fila.get(col, '')))}" for col in columnas_modelo])

        ## Definir Response model
        response_model = build_pydantic_model_dynamic(columnas_modelo)

        ## Procesar
        print(f"  > Procesando {model_id}  - Columnas a validar: {len(columnas_modelo)}")
        response = gpt_call(client, sys_prompt, instr_fase, resp_fase, texto_auditar, respuesta_auditar,  response_model)

        if response:
            dict_res = response.model_dump()
            for key, val in dict_res.items():
                df.at[idx, key] = val
        else:
            for key, val in dict_res.items():
                df.at[idx, key] = "ERROR"  

    df.to_excel(os.path.join(path, "Resultados", "Validación", "Resultados_Validacion.xlsx"), index=False)


Procesando Documento ID: 182...
  > Procesando fase_1  - Columnas a validar: 6
  > Procesando fase_2  - Columnas a validar: 4
  > Procesando fase_3  - Columnas a validar: 6
  > Procesando fase_4  - Columnas a validar: 17


C:\Users\bverg\AppData\Local\Temp\ipykernel_6876\252548389.py:32: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df.at[idx, key] = val
C:\Users\bverg\AppData\Local\Temp\ipykernel_6876\252548389.py:32: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df.at[idx, key] = val
C:\Users\bverg\AppData\Local\Temp\ipykernel_6876\252548389.py:32: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(

  > Procesando fase_5  - Columnas a validar: 3


C:\Users\bverg\AppData\Local\Temp\ipykernel_6876\252548389.py:32: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df.at[idx, key] = val
C:\Users\bverg\AppData\Local\Temp\ipykernel_6876\252548389.py:32: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df.at[idx, key] = val
C:\Users\bverg\AppData\Local\Temp\ipykernel_6876\252548389.py:32: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(

  > Procesando fase_6  - Columnas a validar: 4


C:\Users\bverg\AppData\Local\Temp\ipykernel_6876\252548389.py:32: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df.at[idx, key] = val
C:\Users\bverg\AppData\Local\Temp\ipykernel_6876\252548389.py:32: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df.at[idx, key] = val
C:\Users\bverg\AppData\Local\Temp\ipykernel_6876\252548389.py:32: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(

  > Procesando fase_7  - Columnas a validar: 6


C:\Users\bverg\AppData\Local\Temp\ipykernel_6876\252548389.py:32: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df.at[idx, key] = val
C:\Users\bverg\AppData\Local\Temp\ipykernel_6876\252548389.py:32: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df.at[idx, key] = val
C:\Users\bverg\AppData\Local\Temp\ipykernel_6876\252548389.py:32: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(


Procesando Documento ID: 183...
  > Procesando fase_1  - Columnas a validar: 6
  > Procesando fase_2  - Columnas a validar: 4
  > Procesando fase_3  - Columnas a validar: 6
  > Procesando fase_4  - Columnas a validar: 17
  > Procesando fase_5  - Columnas a validar: 3
  > Procesando fase_6  - Columnas a validar: 4
  > Procesando fase_7  - Columnas a validar: 6

Procesando Documento ID: 184...
  > Procesando fase_1  - Columnas a validar: 6
  > Procesando fase_2  - Columnas a validar: 4
  > Procesando fase_3  - Columnas a validar: 6
  > Procesando fase_4  - Columnas a validar: 17
  > Procesando fase_5  - Columnas a validar: 3
  > Procesando fase_6  - Columnas a validar: 4
  > Procesando fase_7  - Columnas a validar: 6

Procesando Documento ID: 185...
  > Procesando fase_1  - Columnas a validar: 6
  > Procesando fase_2  - Columnas a validar: 4
  > Procesando fase_3  - Columnas a validar: 6
  > Procesando fase_4  - Columnas a validar: 17
  > Procesando fase_5  - Columnas a validar: 3
  > P

### Cálculo de Precisión y Exportación

In [78]:
# --- CÁLCULO DE MÉTRICAS ---
precision_dict = {}
for c in [col for fase in config_fases for col in fase["columnas"]]:
    # Verificamos que la columna exista en el DF para evitar KeyErrors
    if c in df.columns:
        # Filtramos donde realmente había un dato original para validar
        mask_eval = df[c].notna() & (df[c].astype(str).str.strip() != "")
        total = int(mask_eval.sum())
        
        # El status generado por el GPT suele ser {columna}_status
        col_status = f"{c}_status"
        if col_status in df.columns:
            ok = int(((df[col_status] == "OK") & mask_eval).sum())
            precision_dict[c] = 0.0 if total == 0 else round(100.0 * ok / total, 2)
        else:
            precision_dict[c] = 0.0

# 3. Crear DataFrame de resumen y Guardar
df_resumen = pd.DataFrame([precision_dict])
df_resumen["PROMEDIO_GLOBAL"] = df_resumen.mean(axis=1).round(2)

output_file = os.path.join(path, "Resultados", "Validación", "Resultados_Validacion.xlsx")

try:
    with pd.ExcelWriter(output_file, engine="xlsxwriter") as writer:
        df.to_excel(writer, sheet_name="detalle_validacion", index=False)
        df_resumen.to_excel(writer, sheet_name="resumen_precision", index=False)
        
        workbook = writer.book
        worksheet = writer.sheets['detalle_validacion']
        format_wrap = workbook.add_format({'text_wrap': True, 'valign': 'top'})
        
        for i, col in enumerate(df.columns):
            col_len = max(len(str(col)), 10)
            if "comentario" in str(col) or col == "contenido":
                width = 50
            else:
                width = min(col_len + 5, 30)
            worksheet.set_column(i, i, width, format_wrap)

    print(f"\n✅ Proceso completado. Archivo guardado en: {output_file}")
    print("\n--- Resumen de Precisión (%) ---")
    print(df_resumen.T)

except Exception as e:
    print(f"❌ Error al guardar el Excel: {e}")
    df.to_csv("backup_resultados_error.csv", index=False, sep=";")


✅ Proceso completado. Archivo guardado en: C:\Users\bverg\OneDrive - Ministerio de Hacienda\Estudios CNEP - IA CBRs\Resultados\Validación\Resultados_Validacion_GPT.xlsx

--- Resumen de Precisión (%) ---
                                 0
tipo_transaccion            100.00
numero_cbr_actual            98.63
fojas_cbr_actual            100.00
dia_cbr_actual               95.21
mes_cbr_actual               95.21
ano_cbr_actual               95.89
tipo_derecho                 98.63
naturaleza_agua              99.32
ejercicio_derecho            95.21
uso_agua                     95.21
tipo_persona                 99.32
rut                          94.52
nombre                       97.95
nombre_representante_legal   95.59
rut_representante_legal      96.32
direccion                    95.65
tipo_caudal                  96.58
unidad_caudal                97.26
caudal_promedio              90.58
fuente                       81.97
cuenca                       45.83
valor_transaccion         

In [81]:
df_resultados = pd.concat(df_resultados, axis=0)
# Actualizar el dataframe original
for col in df_resultados.columns:
    df.loc[df_resultados.index, col] = df_resultados[col]

print("Proceso finalizado.")

NameError: name 'df_resultados' is not defined